In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
data1=pd.read_csv(r'D:\Navalt\MSC\2023\Noon\noon1.csv')
data2=pd.read_csv(r'D:\Navalt\MSC\2023\Noon\noon2.csv')
data3=pd.read_csv(r'D:\Navalt\MSC\2023\Noon\noon3.csv')

In [3]:
df=pd.concat([data1,data2,data3])
df


,id,pi_sog,pi_stw,true_wind_dir,true_wind_speed,relative_wind_speed,relative_wind_direction,caa,corrected_power,rw,...,rob_garbage_cat_a_plastic,rob_garbage_cat_b_organic,rob_garbage_cat_c_domestic,rob_garbage_cat_e_ashes,qty_sludge_disposed,steaming_time_m_e_lng,reefer_power_consumption,wind_direction_side,sea_direction_side,tot_sludge_removed
0,4055817,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4153647,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3870842,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3862795,0.0,0.0,0.0,0.0,0.0,0.0,0.83101,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4090709,0.0,0.0,0.0,0.0,0.0,0.0,0.83101,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256874,4609309,NaN,NaN,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
256875,4609310,NaN,NaN,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
256876,4609311,NaN,NaN,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
256877,4609314,NaN,NaN,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
year=2023

In [5]:
df['report_date_time'] = pd.to_datetime(df['report_date_time'])
df = df.sort_values(by='report_date_time')
df['year'] = df['report_date_time'].dt.year

df = df[df['year']==year]

df.columns.to_list()


In [22]:
import pandas as pd
import numpy as np

def calculate_total_fuel_consumption(df):
    df['hfo'] = df[['fuel_me_rsdl_hs', 'fuel_aux_rsdl_hs', 'fuel_boiler_rsdl_hs']].sum(axis=1)
    df['lfo'] = df[["fuel_me_rsdl_vls", "fuel_me_rsdl_uls", "fuel_aux_rsdl_vls", "fuel_aux_rsdl_uls",
                    "fuel_boiler_rsdl_vls", "fuel_boiler_rsdl_uls"]].sum(axis=1)
    df['mgo'] = df[["fuel_me_dstlt_vls", "fuel_me_dstlt_uls", "fuel_me_tnktnr_dstlt_vls", "fuel_aux_dstlt_vls",
                    "fuel_aux_dstlt_uls", "fuel_aux_tnktnr_dstlt_vls", "fuel_boiler_dstlt_vls",
                    "fuel_boiler_dstlt_uls", "fuel_boiler_tnktnr_dstlt_vls"]].sum(axis=1)

    df['total_fuel'] = df[['hfo', 'lfo', 'mgo']].sum(axis=1)

    fuel_columns = ['hfo', 'lfo', 'mgo']
    for column in fuel_columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

    df['total_fuel_consumption'] = df[fuel_columns].sum(axis=1)

    df['total_fuel_consumption_at_sea'] = df[df['status'] == 'AT SEA'][fuel_columns].sum(axis=1)

    df['total_fuel_consumption_at_berth'] = df[(df['status'] == 'DRIFTING') | (df['status'] == 'IN PORT')][
        fuel_columns].sum(axis=1)

    return df

def calculate_total_co2_emission(df):
    df['total_co2'] = (
        pd.to_numeric(df['hfo']) * 3.114 +
        pd.to_numeric(df['lfo']) * 3.151 +
        pd.to_numeric(df['mgo']) * 3.206
    )

    df['total_co2_sea'] = df[df['status'] == 'AT SEA']['total_co2']
    df['total_co2_berth'] = df[(df['status'] == 'DRIFTING') | (df['status'] == 'IN PORT')][
        'total_co2']
    return df

def calculate_total_time_sea(sea):
    sea['me_fuel_only_steaming_time'] = sea['me_fuel_only_steaming_time'].replace({np.nan: None})
    total_time_sea = sea['me_fuel_only_steaming_time'].sum()
    total_time_sea = total_time_sea / 24 
    return total_time_sea

def calculate_total_cargo(df):
    total_cargo_ton = df['cargo_total'].iloc[0]  
    total_cargo_teu = df['cargo_total_teu'].iloc[0]  
    return {"ton": total_cargo_ton, "teu": total_cargo_teu}

def prepare_report_data(df, total_cargo, total_time_sea):
    total_distance_sailed = df['miles_by_gps'].astype(float).sum() + df['manvrng_miles_by_gps'].astype(float).sum()

    data = {
        "Total Fuel Consumption": round(df['total_fuel_consumption'].sum(), 2),
        "Total Fuel Consumption at Sea (Ton)": round(df['total_fuel_consumption_at_sea'].sum(), 2),
        "Total Fuel Consumption at Berth (Ton)": round(df['total_fuel_consumption_at_berth'].sum(), 2),
        "Total CO2 Emission (Ton)": round(df['total_co2'].sum(), 2),
        "CO2 Emissions at Sea (Ton)": round(df['total_co2_sea'].sum(), 2),
        "CO2 Emission at Berth (Ton)": round(df['total_co2_berth'].sum(), 2),
        "Total Distance Sailed (Nm)": round(total_distance_sailed, 2),
        "Total time at sea (days)": round(total_time_sea),
        "Total Cargo (Ton)": round(total_cargo["ton"], 2),
        "Total Cargo (TEU)": round(total_cargo["teu"], 2)
    }
    return data

def get_unique_voyage_orders(df, is_eu=False):
    if is_eu:
        filtered_df = df[df['eu'] == 'Y']
    else:
        filtered_df = df[df['eu'] == 'N']
    return filtered_df['voyage_order'].unique()

def display_voyage_orders(voyage_orders):
    print("Available voyage orders:")
    for order in voyage_orders:
        print(order)

def display_report_for_voyage_order(df, types="All voyages", start_date=None, end_date=None, vessel=None):
    if types == "All voyages":
        filtered_df = df.copy()
    elif types == "EU voyages":
        filtered_df = df[df['eu'] == 'Y']
    else:
        print("Invalid type. Please specify 'All voyages' or 'EU voyages'.")
        return
    
    if start_date and end_date:
        filtered_df = filtered_df[(filtered_df['report_date_time'] >= start_date) & 
                                  (filtered_df['report_date_time'] <= end_date)]
    
    if vessel:
        filtered_df = filtered_df[filtered_df['vessel'] == vessel]
    
    voyage_orders = get_unique_voyage_orders(filtered_df)
    display_voyage_orders(voyage_orders)
    
    while True:
        voyage_order_input = input("Enter the voyage order: ")
        try:
            voyage_order = float(voyage_order_input)  # Convert input to float
            if voyage_order in voyage_orders:
                selected_data = filtered_df[filtered_df['voyage_order'] == voyage_order]
                calculate_total_fuel_consumption(selected_data)  
                calculate_total_co2_emission(selected_data)  
                total_time_sea = calculate_total_time_sea(selected_data)
                total_cargo = calculate_total_cargo(selected_data)
                report_data = prepare_report_data(selected_data, total_cargo, total_time_sea)
                
                voyage_code = selected_data['voyage_code'].iloc[0]
                
                print(f"Selected Voyage Code: {voyage_code}, Voyage Order: {voyage_order}")
                return report_data
            
            else:
                print("Invalid voyage order. Please try again.")
        except ValueError:
            print("Invalid input. Please enter a numeric voyage order.")

# Example usage
viewer = df  
types = "All voyages"
start_date = "2023-01-01" 
end_date = "2023-04-01"
vessel = "MSC ADITI"
selected_report_data = display_report_for_voyage_order(viewer, types=types, start_date=start_date, end_date=end_date, vessel=vessel)
selected_report_data



Available voyage orders:
610.0


KeyboardInterrupt: Interrupted by user